# 05 — Accelerated fixed-point search and the sweep CLI

Two ways to scale up:
- `find_all_fixed_points(prefilter_threshold=..., parallel=True)` —
  batch β-evaluation prefilter + ProcessPoolExecutor refinement.
- `asymsafety eval` / `asymsafety scan` — reproducible parameter studies
  driven from the shell, with `.npz`/`.h5`/`.json` output.


In [ ]:
import time
import subprocess
import sys
import numpy as np
from asymsafety.beta.matter import build_gravity_matter_fp_system
from asymsafety.actions.matter import MatterContent
from asymsafety.analysis.fixed_points import FixedPointFinder
from asymsafety.cli.io import read_npz


## 1. Serial vs. parallel `find_all_fixed_points`


In [ ]:
system = build_gravity_matter_fp_system(
    MatterContent(n_scalars=1, n_dirac=1),
    scalar_quartic=True, yukawa=True, running_xi=True,
)
finder = FixedPointFinder(system)

t0 = time.perf_counter()
fps_serial = finder.find_all_fixed_points(n_grid=4, n_random=200)
t_serial = time.perf_counter() - t0

t0 = time.perf_counter()
fps_par = finder.find_all_fixed_points(n_grid=4, n_random=200, parallel=True, max_workers=4)
t_parallel = time.perf_counter() - t0

print(f'Serial:   {t_serial:.2f}s, found {len(fps_serial)} FPs')
print(f'Parallel: {t_parallel:.2f}s, found {len(fps_par)} FPs')
print(f'Speedup:  {t_serial/t_parallel:.1f}x')


## 2. The `asymsafety eval` CLI

Same batch evaluation, driven from the shell. Output is a portable
.npz file you can share or post-process.


In [ ]:
subprocess.run(
    [sys.executable, '-m', 'asymsafety.cli.main', 'eval',
     '--truncation', 'eh',
     '--grid', 'g:0:1.5:30,lambda:-0.4:0.4:30',
     '--output', '/tmp/eh_grid.npz'],
    check=True,
)


## 3. Load the result and visualize


In [ ]:
import matplotlib.pyplot as plt
arrays, meta = read_npz('/tmp/eh_grid.npz')
g_axis = arrays['axis_g']
lam_axis = arrays['axis_lambda']
betas = arrays['betas']  # shape (N_total, 2)
norms = np.linalg.norm(betas, axis=1).reshape(len(g_axis), len(lam_axis))

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(np.log10(norms.T + 1e-12),
                origin='lower', aspect='auto',
                extent=[g_axis[0], g_axis[-1], lam_axis[0], lam_axis[-1]],
                cmap='viridis')
ax.set_xlabel('g'); ax.set_ylabel('λ')
ax.set_title('log₁₀ |β| over the (g, λ) grid')
plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()


Dark spots in the heatmap mark fixed points (where |β| → 0). The CLI
produces the same data structure that `find_all_fixed_points` operates
on internally, so any downstream analysis can use either entry point.
